# Linear Regression from scratch

This notebook mirrors the ML Math Lab Implementation lesson. It uses a deterministic toy housing dataset, standard Python and NumPy, then compares the result with scikit-learn. Run the cells from top to bottom in Google Colab or a local Jupyter environment. No credentials or paid services are required.

## 1. Features and targets
Each position describes one observation. Floor area is measured in thousands of square feet and price in thousands of dollars.

In [ ]:
import random
import numpy as np

random.seed(7)
np.random.seed(7)
xs = [1.2, 1.8, 2.5, 3.1, 3.8, 4.2]
ys = [178.0, 213.0, 246.0, 278.0, 316.0, 349.0]
assert len(xs) == len(ys)
list(zip(xs, ys))

## 2. Prediction, residuals and metrics
We use the course convention `residual = prediction - actual`. MAE averages absolute residuals. MSE averages squared residuals.

In [ ]:
def predict_one(x, w, b):
    return w * x + b

def predict_all(features, w, b):
    return [predict_one(x, w, b) for x in features]

def residuals(predictions, targets):
    return [prediction - actual for prediction, actual in zip(predictions, targets)]

def mae(errors):
    return sum(abs(error) for error in errors) / len(errors)

def mse(errors):
    return sum(error ** 2 for error in errors) / len(errors)

initial_predictions = predict_all(xs, w=42.0, b=140.0)
initial_errors = residuals(initial_predictions, ys)
print('predictions:', initial_predictions)
print('residuals:', initial_errors)
print('MAE:', mae(initial_errors), 'MSE:', mse(initial_errors), 'RMSE:', mse(initial_errors) ** 0.5)

### Exercise A — predict before running
What does `predict_one(2.5, 42, 140)` return? Expected outcome: `245.0`.

Common mistakes: reversing the residual sign, averaging signed residuals, or forgetting to square each residual before averaging MSE.

In [ ]:
exercise_a = predict_one(2.5, 42.0, 140.0)
assert exercise_a == 245.0
exercise_a

## 3. MSE gradients and one update
For `m` observations, `dw = (2/m) Σ(residual × x)` and `db = (2/m) Σ residual`. Gradient descent subtracts the learning-rate-scaled gradient.

In [ ]:
def mse_gradients(features, errors):
    m = len(features)
    dw = (2 / m) * sum(error * x for error, x in zip(errors, features))
    db = (2 / m) * sum(errors)
    return dw, db

def update(w, b, dw, db, learning_rate):
    return w - learning_rate * dw, b - learning_rate * db

tiny_xs, tiny_ys = [1.0, 2.0], [3.0, 5.0]
tiny_w, tiny_b = 1.0, 0.0
tiny_predictions = predict_all(tiny_xs, tiny_w, tiny_b)
tiny_errors = residuals(tiny_predictions, tiny_ys)
tiny_dw, tiny_db = mse_gradients(tiny_xs, tiny_errors)
tiny_w, tiny_b = update(tiny_w, tiny_b, tiny_dw, tiny_db, 0.1)
assert np.isclose(tiny_w, 1.8) and np.isclose(tiny_b, 0.5)
tiny_w, tiny_b

### Exercise B — debug the update
If code uses `w + learning_rate * dw`, replace `+` with `-`. Expected updated parameters for the tiny dataset are `w=1.8`, `b=0.5`. Adding the gradient performs gradient ascent.

## 4. Training loop
Iteration 0 is the initial parameter state. Every loop below creates predictions, computes residuals and gradients, then updates both parameters.

In [ ]:
def fit_linear_regression(features, targets, learning_rate=0.01, steps=4000):
    w, b = 0.0, 0.0
    history = []
    for step in range(steps):
        predictions = predict_all(features, w, b)
        errors = residuals(predictions, targets)
        history.append(mse(errors))
        dw, db = mse_gradients(features, errors)
        w, b = update(w, b, dw, db, learning_rate)
    return w, b, history

scratch_w, scratch_b, history = fit_linear_regression(xs, ys)
assert history[-1] < history[0]
print('from scratch:', scratch_w, scratch_b, 'final MSE:', history[-1])

## 5. Unseen and multiple-feature predictions
Inference uses learned parameters without updating them. Multiple features replace `w*x` with a dot product.

In [ ]:
unseen_area = 3.4
unseen_prediction = predict_one(unseen_area, scratch_w, scratch_b)

def predict_many(features, weights, bias):
    if len(features) != len(weights):
        raise ValueError('features and weights must have the same length')
    return sum(x * w for x, w in zip(features, weights)) + bias

multi_prediction = predict_many([2.0, 3.0], [30.0, -2.0], 10.0)
assert multi_prediction == 64.0
unseen_prediction, multi_prediction

## 6. Compare with scikit-learn
Fit both implementations to the same rows. In a real project, split before fitting and keep test rows untouched until evaluation.

In [ ]:
from sklearn.linear_model import LinearRegression

X = np.array(xs, dtype=float).reshape(-1, 1)
y = np.array(ys, dtype=float)
model = LinearRegression().fit(X, y)
sklearn_predictions = model.predict(X)
scratch_predictions = np.array(predict_all(xs, scratch_w, scratch_b))
print('scikit-learn coefficient/intercept:', model.coef_[0], model.intercept_)
print('maximum prediction difference:', np.max(np.abs(sklearn_predictions - scratch_predictions)))
assert np.max(np.abs(sklearn_predictions - scratch_predictions)) < 0.1

## 7. Practice and expected outcomes
1. Change the initial weight in the tiny example and predict the residual signs before running.
2. Replace MSE with MAE and explain why the MSE gradient formula no longer applies.
3. Create a train/test split before fitting. Expected workflow: fit only on training rows, then compute metrics on untouched test rows.
4. Deliberately use a very large learning rate and observe instability. Restore `0.01` afterward.

Common mistakes to watch for: mismatched feature/target lengths, reversed residual sign, averaging before squaring, adding the gradient, updating `w` but not `b`, evaluating on training data only, and fitting preprocessing with test data.